# S04 - Volatility-managed exposure (volatility)

**Hypothesis.** Volatility clusters and is persistent, but high volatility is not followed by
proportionally higher returns. Scaling equity exposure down when recent realised volatility is
high should raise the Sharpe ratio (Moreira & Muir 2017). Here without leverage: the weight is
capped at 100% equities, the rest earns T-bills.

| Variant | Equity weight at each monthly check |
|---|---|
| lb21_t10, lb21_t15 | min(1, 10% or 15% / realised vol of the last 21 trading days) |
| lb63_t10, lb63_t15 | same with a 63-day window |

**Inputs and point-in-time lags**

| Series | Source | Lag used |
|---|---|---|
| S&P 500 daily total return | `bt_long.long_sample` (spliced Shiller / ^SP500TR) | realised vol uses returns up to the decision close; next-close fill |

Monthly check, next-close execution, 5 bp per unit of turnover (fractional weight changes).

**Sources.** Moreira & Muir (2017), *Volatility-managed portfolios*, Journal of Finance,
NBER w22208; Cederburg, O'Doherty, Wang & Yan (2020), *On the performance of
volatility-managed portfolios*, JFE doi:10.1016/j.jfineco.2020.04.015 (out-of-sample doubts);
Harvey et al. (2018), *The impact of volatility targeting*, SSRN 3175538.

> Decision-support research only; never places orders.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
sys.path.insert(0, str(_root / "notebooks" / "backtest"))
import bt_common as bc  # noqa: E402
import bt_long  # noqa: E402
import bt_periods as bp  # noqa: E402
import bt_pit  # noqa: E402
import bt_registry as br  # noqa: E402
import bt_stats as bs  # noqa: E402
import seed_registry  # noqa: E402
import stage2 as s2  # noqa: E402

bc.setup_plotly("cdn")
pd.set_option("display.width", 180)
TRIAL_ID, STRATEGY, NOTEBOOK = 'S04', 'vol_managed', 'volatility/vol_managed.ipynb'
PROTOCOL, REBALANCE = 'main', 'monthly'

sample = bt_long.long_sample()
px, cash = sample["px"], sample["cash"]
bench = bp.benchmark_returns(px, cash)  # SMA200 monthly / B&H / 60-40 / T-bills, frozen settings
print("prices", px.index[0].date(), "->", px.index[-1].date(), "| registry trials so far:", br.trial_count())

prices 1934-02-01 -> 2026-10-19 | registry trials so far: 13


## 1. Data and point-in-time inputs

In [2]:
inputs = {"px": px}
r = px["tr_close"].pct_change(fill_method=None)
print("realised vol (21d, annualised), last:", round(float(r.tail(21).std() * np.sqrt(252)), 4))

realised vol (21d, annualised), last: 0.1039


## 2. Signal and variant grid (fixed before any backtest)

In [3]:
def build_signal(inputs, lb=21, target=0.10):
    r = inputs["px"]["tr_close"].pct_change(fill_method=None)
    vol = r.rolling(lb).std() * np.sqrt(252)
    return (target / vol).clip(upper=1.0).where(vol.notna())


VARIANTS = {
    f"lb{lb}_t{int(t * 100)}": dict(lb=lb, target=t) for lb in (21, 63) for t in (0.10, 0.15)
}
PARAMS_TEXT = {n: f"{n}; weight=min(1,target/realised vol); monthly; next_close; 5bp/turnover" for n in VARIANTS}

In [4]:
w = build_signal(inputs, lb=63, target=0.10)
fig = go.Figure()
fig.add_trace(go.Scatter(x=w.index, y=w.rolling(21).mean(), name="weight lb63_t10 (21d avg)"))
fig.update_layout(title="Volatility-managed equity weight", height=320, yaxis_tickformat=".0%")
fig.show()

## 3. Look-ahead checks

Every variant is recomputed with all inputs cut at past dates (monthly inputs: completed months only) and must equal the full-data signal on those dates (`stage2.truncation_test`). No backtest runs unless all pass.

In [5]:
N_CUTS = 30
trunc = {}
for name, params in VARIANTS.items():
    trunc[name] = s2.truncation_test(build_signal, inputs, n_cuts=N_CUTS, **params)
display(pd.DataFrame({k: v.row() for k, v in trunc.items()}).T)
bad = [k for k, v in trunc.items() if not v.passed]
assert not bad, f"look-ahead detected in variants {bad}: fix before any backtest"
print("all variants pass the truncation look-ahead test")

,cuts,mismatches,max_abs_diff,passed
lb21_t10,30,0,0.0,True
lb21_t15,30,0,0.0,True
lb63_t10,30,0,0.0,True
lb63_t15,30,0,0.0,True


all variants pass the truncation look-ahead test


## 4. Development-period sweep

Each variant is scored on the **dev** period only and registered as a trial. The best dev Sharpe is chosen; validation and holdout results are never used for the choice.

In [6]:
signals = {name: build_signal(inputs, **params) for name, params in VARIANTS.items()}
sweep, CHOSEN = s2.dev_sweep(px, cash, bench, signals, PARAMS_TEXT, trial_id=TRIAL_ID, strategy=STRATEGY,
                             notebook=NOTEBOOK, protocol=PROTOCOL, rebalance=REBALANCE)
display(sweep)
print("chosen on dev Sharpe only:", CHOSEN, "|", PARAMS_TEXT[CHOSEN])

,dev_sharpe,d_sma200,d_bh,time_in_market,switches_pa,dev_start
lb21_t10,0.56455,-0.079186,0.053551,0.800601,9.051411,1935-03-13
lb21_t15,0.570718,-0.073018,0.059719,0.929125,4.379715,1935-03-13
lb63_t10,0.534742,-0.108994,0.023743,0.783905,8.877731,1935-05-13
lb63_t15,0.536332,-0.107403,0.025334,0.925455,3.935489,1935-05-13


chosen on dev Sharpe only: lb21_t15 | lb21_t15; weight=min(1,target/realised vol); monthly; next_close; 5bp/turnover


## 5. Frozen evaluation of the chosen variant

Frozen primary metric (`PLAN.md` section 2): beat SMA200 and buy & hold in dev **and** val, one-sided block-bootstrap p <= 0.10 vs SMA200 in both, deflated Sharpe >= 0.95 with the registry trial count. Holdout is confirmatory only.

In [7]:
signal = signals[CHOSEN]
runs, cmp, stats, res = s2.evaluate(px, cash, bench, signal, protocol=PROTOCOL, rebalance=REBALANCE)
display(cmp.round(3))
display(stats)
display(res.to_frame().round(3))
print(f"n_trials={res.n_trials} | DSR={res.dsr:.3f} | passes={res.passes} | holdout confirms={res.holdout_confirms}")
display(pd.Series(res.checks, name="check"))

,strategy,bh,d_bh,sma200,d_sma200,mix60,d_mix60
dev,0.571,0.511,0.060,0.644,-0.073,0.518,0.053
val,0.383,0.319,0.064,0.572,-0.189,0.309,0.074
holdout,0.794,0.781,0.013,0.658,0.136,0.779,0.015
full,0.571,0.513,0.058,0.630,-0.059,0.515,0.056


,CAGR,Vol,Sharpe,Sortino,Max DD,Calmar,Worst day,MaxDD peak,MaxDD recovered,Final x,Time in market,Switches,Switches / yr,Cost drag / yr
dev,0.110128,0.131875,0.570718,0.788648,-0.48823,0.225567,-0.172457,1937-03-10,1944-07-05,306.454973,0.929125,240,4.379715,0.000389
val,0.083171,0.136987,0.382842,0.537907,-0.410532,0.202594,-0.063487,2000-09-01,2006-10-04,4.940275,0.882113,129,6.451766,0.000496
holdout,0.122227,0.139502,0.793545,1.104629,-0.247277,0.494294,-0.074511,2020-02-19,2021-01-07,6.930774,0.906457,93,5.539506,0.000499
full,0.106356,0.134427,0.571065,0.793217,-0.48823,0.21784,-0.172457,1937-03-10,1944-07-05,10492.99645,0.91467,462,5.043502,0.000433


,Sharpe,d_sma200,d_bh,p_vs_sma200
dev,0.571,-0.073,0.060,0.847
val,0.383,-0.189,0.064,0.921
holdout,0.794,0.136,0.013,NaN
full,0.571,-0.059,0.058,NaN


n_trials=17 | DSR=1.000 | passes=False | holdout confirms=True


dev_beats_sma200        False
dev_beats_bh             True
dev_p_le_0.1            False
val_beats_sma200        False
val_beats_bh             True
val_p_le_0.1            False
dsr_ge_0.95              True
holdout_d_sma200_pos     True
Name: check, dtype: bool

In [8]:
s2.plot_growth(runs, bench, f'{TRIAL_ID} {CHOSEN}', PROTOCOL).show()

## 6. Robustness: costs, placebo, peek test, decades

In [9]:
cs = bs.cost_sensitivity(px, signal, cash, rebalance=REBALANCE)
display(cs.round(3))
sma_full = bs.excess_sharpe(bench["sma200"].reindex(runs["full"].index), bench["tbill"].reindex(runs["full"].index))
be = bs.breakeven_cost_bps(px, signal, cash, bench_sharpe=sma_full, rebalance=REBALANCE)
print(f"break-even one-way cost vs SMA200 Sharpe over the same window ({sma_full:.3f}): {be:.1f} bp")
pl = bs.placebo_test(px, signal, cash, n_placebo=200, rebalance=REBALANCE, rng=np.random.default_rng(bs.SEED))
display(pd.Series(pl, name="placebo").to_frame())
peek = bt_pit.lookahead_test(px, signal, cash, rebalance=REBALANCE)
print(f"peek test: Sharpe by shift {peek.sharpe_by_shift.round(3).to_dict()} | gain {peek.peek_gain:+.3f}")
display(s2.decade_table(runs["full"], bench).style.format("{:.2f}"))

,Sharpe,CAGR,Max DD,turnover_pa
cost_bps,,,,
0,0.554,0.104,-0.486,0.879
5,0.551,0.103,-0.488,0.879
10,0.548,0.103,-0.491,0.879
25,0.538,0.101,-0.498,0.879
50,0.522,0.099,-0.511,0.879


break-even one-way cost vs SMA200 Sharpe over the same window (0.630): 0.0 bp


,placebo
stat,Sharpe
real,0.551186
placebo_mean,0.498549
placebo_q95,0.542054
quantile,0.98
p_one_sided,0.02
n_placebo,200
time_in_market,0.950901


peek test: Sharpe by shift {-21: 0.793, -5: 0.577, -1: 0.553, 0: 0.551, 1: 0.547, 5: 0.531, 21: 0.503} | gain +0.241


,Sharpe,Sharpe B&H,Sharpe SMA200,CAGR,CAGR B&H,CAGR SMA200,time in market
1930s,0.61,0.26,0.41,0.10,0.03,0.05,0.66
1940s,0.52,0.61,0.57,0.07,0.09,0.06,0.93
1950s,1.42,1.42,1.51,0.19,0.19,0.18,0.98
1960s,0.41,0.40,0.43,0.08,0.08,0.07,0.98
1970s,0.04,0.03,0.37,0.06,0.06,0.09,0.95
1980s,0.55,0.53,0.58,0.17,0.17,0.17,0.93
1990s,0.87,0.92,0.64,0.16,0.18,0.13,0.94
2000s,-0.04,-0.05,0.50,0.01,-0.01,0.08,0.82
2010s,0.89,0.90,0.71,0.12,0.14,0.09,0.93
2020s,0.68,0.67,0.58,0.13,0.15,0.10,0.88


## 7. Register the result

In [10]:
note = ("passes" if res.passes else "fails") + f" frozen metric; chosen {CHOSEN} on dev; protocol {PROTOCOL}"
rows = s2.register_final(TRIAL_ID, STRATEGY, NOTEBOOK, PARAMS_TEXT[CHOSEN], res, note=note, header=seed_registry.HEADER)
display(rows)
print("registry trial count:", br.trial_count())

,trial_id,date,strategy,notebook,params,period,sharpe,d_sma200,d_bh,p_vs_sma200,dsr,n_trials,passes,status,note
42,S04-v1/dev,2026-10-20,vol_managed,volatility/vol_managed.ipynb,"lb21_t10; weight=min(1,target/realised vol); m...",dev,0.5645,-0.0792,0.0536,,,,,run,"dev-only sweep, variant lb21_t10"
43,S04-v2/dev,2026-10-20,vol_managed,volatility/vol_managed.ipynb,"lb21_t15; weight=min(1,target/realised vol); m...",dev,0.5707,-0.0730,0.0597,,,,,run,"dev-only sweep, variant lb21_t15"
44,S04-v3/dev,2026-10-20,vol_managed,volatility/vol_managed.ipynb,"lb63_t10; weight=min(1,target/realised vol); m...",dev,0.5347,-0.1090,0.0237,,,,,run,"dev-only sweep, variant lb63_t10"
45,S04-v4/dev,2026-10-20,vol_managed,volatility/vol_managed.ipynb,"lb63_t15; weight=min(1,target/realised vol); m...",dev,0.5363,-0.1074,0.0253,,,,,run,"dev-only sweep, variant lb63_t15"
46,S04/dev,2026-10-20,vol_managed,volatility/vol_managed.ipynb,"lb21_t15; weight=min(1,target/realised vol); m...",dev,0.5707,-0.0730,0.0597,0.8470,,17,False,fail,fails frozen metric; chosen lb21_t15 on dev; p...
47,S04/val,2026-10-20,vol_managed,volatility/vol_managed.ipynb,"lb21_t15; weight=min(1,target/realised vol); m...",val,0.3828,-0.1894,0.0638,0.9208,,17,False,fail,fails frozen metric; chosen lb21_t15 on dev; p...
48,S04/holdout,2026-10-20,vol_managed,volatility/vol_managed.ipynb,"lb21_t15; weight=min(1,target/realised vol); m...",holdout,0.7935,0.1360,0.0128,,,17,False,fail,fails frozen metric; chosen lb21_t15 on dev; p...
49,S04/full,2026-10-20,vol_managed,volatility/vol_managed.ipynb,"lb21_t15; weight=min(1,target/realised vol); m...",full,0.5711,-0.0587,0.0577,,0.9998,17,False,fail,fails frozen metric; chosen lb21_t15 on dev; p...


registry trial count: 17


## 8. Current signal

In [11]:
last = signal.dropna()
print(f"latest signal ({CHOSEN}) on {last.index[-1].date()}: equity weight {last.iloc[-1]:.2f}")
chk = bc.check_days(px.index, REBALANCE)
display(last[chk.reindex(last.index).fillna(False)].tail(6).to_frame("equity weight at decision"))
print("Research output only -- not a recommendation or an order.")

latest signal (lb21_t15) on 2026-10-19: equity weight 1.00


,equity weight at decision
Date,
2026-04-30,1.000000
2026-05-29,1.000000
2026-06-30,0.866006
2026-07-31,1.000000
2026-08-31,1.000000
2026-09-30,1.000000


Research output only -- not a recommendation or an order.


## 9. Evidence summary (computed)

In [12]:
for line in s2.evidence_lines(res, trunc[CHOSEN], peek, pl, be):
    print("*", line)

* Truncation look-ahead test: PASSED on 30 cut dates; peek test gain +0.241.
* dev: Sharpe 0.571 (vs SMA200 -0.073, vs B&H +0.060, p vs SMA200 0.847)
* val: Sharpe 0.383 (vs SMA200 -0.189, vs B&H +0.064, p vs SMA200 0.921)
* holdout: Sharpe 0.794 (vs SMA200 +0.136, vs B&H +0.013)
* Deflated Sharpe 1.000 with n_trials=17.
* Placebo: real 0.551 vs placebo 95th pct 0.542 (p 0.020); break-even cost vs SMA200 full 0.0 bp.
* Verdict: FAILS the frozen primary metric (failed checks: dev_beats_sma200, dev_p_le_0.1, val_beats_sma200, val_p_le_0.1); holdout confirms.


## 10. Caveats

* Fractional weights: the placebo test binarises the signal at 50%, so it is only indicative here.
* No leverage: the published gains of volatility management rely partly on levering up in calm
  markets; capped at 100% the strategy can only de-risk.
* Monthly rebalancing ignores intra-month volatility spikes (e.g. 1987, 2020).

## 11. Interpretation

* Volatility management **improves on buy & hold in every period** (Sharpe +0.06 in dev and val, +0.01 in
  the holdout) with a smaller drawdown (-49% vs -55%), consistent with Moreira & Muir. But it trails
  SMA200 by 0.07 in dev and 0.19 in val (bootstrap p 0.85 / 0.92), so it **fails** the frozen metric.
* Without leverage the rule can only cut exposure; the dev sweep preferred the milder 15% target
  (93% average weight), i.e. it de-risks only in the most turbulent months. It does not avoid slow
  bear markets with moderate volatility (2000-02, 2007-08 before Lehman), which is where SMA200 earns
  most of its edge (2000s decade: Sharpe -0.04 vs SMA200 0.50).
* Holdout (2010+): Sharpe 0.79, the best of S02-S04 and above SMA200 (0.66), but holdout is
  confirmatory only and the binarised placebo (p 0.02) is only indicative for fractional weights.
* Practical reading: vol targeting is a risk-reduction overlay, not a timing signal. A combination
  with SMA200 (trend for slow bears, vol for fast crashes) is a natural candidate for an ensemble.